# step-counter-increment — ex3: max_steps early termination breaks both inner and outer loop

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `step-counter-increment`. Running the final beacon cell reports progress against the `Trainer: step counter increment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: step counter increment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`step-counter-increment`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "step-counter-increment"
DD_SUBTOPIC = "Trainer: step counter increment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `max_steps` termination — train by steps, not just by epochs

Ex1 + ex2 incremented a step counter and gated logging on it. The deepening move uses the SAME counter as a STOP condition: training halts as soon as `step >= max_steps`, even mid-epoch.

```python
step = 0
stopped = False
for epoch in range(n_epochs):
    for batch in train_loader:
        # ... forward, backward, optimizer.step() ...
        step += 1
        if step >= max_steps:
            stopped = True
            break
    if stopped:
        break
```

**Why step-based termination is standard for LLM training.** Epochs are dataset-size-dependent — the same 'epoch' is 10x more compute on a 10x larger dataset. Step-based budgets (`max_steps=100_000`) are dataset-independent. Hugging Face's `TrainingArguments.max_steps` uses exactly this pattern.

**Two breaks: one inner, one outer.** Python's `break` only exits the INNERMOST loop. The outer `for epoch` needs its own break gated on a `stopped` flag — otherwise you'd inadvertently start a new epoch.

**Check AFTER the tick.** `step += 1` then `if step >= max_steps: break`. Doing the check BEFORE the tick (or not ticking at all) would either off-by-one the final step count or skip the last batch. Tick-then-check is the safe order.

### Exercise 3 — max_steps early termination breaks both inner and outer loop

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply step-counter-driven early termination — `if step >= max_steps: break` after the counter tick — and propagate the break out of the nested epoch loop using a `stopped` flag.
> Keywords: max-steps, early-stop, training-budget, break
> ```

**KCs targeted:** `max-steps-checked-after-counter-tick`, `nested-loop-break-via-stopped-flag`

Implement `ex3_train_until_max_steps(losses_per_epoch, max_steps)`. A training-loop driver where the outer loop walks epochs and the inner loop walks batches, but training halts as soon as the step counter reaches `max_steps` — even mid-epoch.

Inputs:
- `losses_per_epoch`: `list[list[float]]`. Each inner list is one epoch's per-batch losses.
- `max_steps`: int — global step budget.

Return a dict:
- `'final_step'`: int — the step counter's final value.
- `'logged_losses'`: `list[float]` — every loss the function processed before stopping, in order.
- `'stopped_early'`: bool — True iff training halted before all epochs were consumed.

Loop structure:
1. `step = 0`, `stopped = False`, `logged = []`.
2. For each `epoch_losses` in `losses_per_epoch`:
   - For each `loss` in `epoch_losses`:
     - `logged.append(loss)`
     - `step += 1`
     - If `step >= max_steps`: set `stopped = True`, `break`.
   - If `stopped`: `break` (out of the epoch loop too).
3. Return the dict.

**Tick BEFORE the check.** The order is `step += 1` then `if step >= max_steps: break`, so the final `step` value equals `min(max_steps, total_batches)`.

In [ ]:
def ex3_train_until_max_steps(losses_per_epoch, max_steps):
    step = 0
    stopped = False
    logged = []
    for epoch_losses in losses_per_epoch:
        for loss in epoch_losses:
            logged.append(loss)
            step += 1
            if step >= max_steps:
                stopped = True
                break
        if stopped:
            break
    # If we exited because epochs ran out (not because step hit max), stopped_early stays False.
    return {
        'final_step': step,
        'logged_losses': logged,
        'stopped_early': stopped,
    }


<details><summary>Solution</summary>

```python
def ex3_train_until_max_steps(losses_per_epoch, max_steps):
    step = 0
    stopped = False
    logged = []
    for epoch_losses in losses_per_epoch:
        for loss in epoch_losses:
            logged.append(loss)
            step += 1
            if step >= max_steps:
                stopped = True
                break
        if stopped:
            break
    # If we exited because epochs ran out (not because step hit max), stopped_early stays False.
    return {
        'final_step': step,
        'logged_losses': logged,
        'stopped_early': stopped,
    }
```

**Two `break`s are required.** Python's `break` only exits the innermost loop, so the inner break ends the current epoch, then the outer `if stopped: break` ends the epoch loop. Forgetting the outer one means training resumes in the next epoch — silently doing extra work.

**Tick-then-check is the safe order.** `step += 1` followed by `if step >= max_steps: break` makes `final_step == min(max_steps, total_batches)`. Reversing the order (check then tick) would off-by-one your reporting.

**Hugging Face's pattern.** `TrainingArguments.max_steps` uses exactly this break-twice pattern in `Trainer._inner_training_loop`. For LLM training, step-based budgets (compute-bound) are far more common than epoch-based budgets (dataset-size-bound).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()